In [ ]:
# import marimo as mo

# Mapping Enviroweather Stations

This is an abbreviated demonstration of how to get a list of stations, make a maps and select one to show details.

In [1]:
# imports
import requests
import json
from datetime import datetime, date
from os import path


setup

In [2]:


todays_date = date.today().isoformat()
todays_date

'2026-09-29'

### Configuration

In [3]:
# alternatively read JSON from disk with multiple environments in it

config_json = """
{
    "production": {
        "api_url": "https://api.enviroweather.msu.edu/ewx-api/api",
        "rm_api_url": "https://api.enviroweather.msu.edu/ewx-api/api"
    }
}
"""

# request file ewx_environments.json for full list of environments
# some environments require MSU VPN access
config_file = 'ewx_environments.json'
# with open(config_file,"r") as f: ewx_env_dict  = json.load(f)

ewx_env_dict = json.loads(config_json)

ewx_env = 'production'
api_url = ewx_env_dict[ewx_env]["api_url"]
rm_api_url = ewx_env_dict[ewx_env]["rm_api_url"]

api_url

'https://api.enviroweather.msu.edu/ewx-api/api'

In [ ]:
# confirm urls exist

try:
    response = requests.options(path.dirname(api_url))
    if response.ok: 
        url_check_msg = "Success - API is accessible."
    else:
        url_check_msg = response.status_code
except Exception as e:
     url_check_msg  = f"Failure to connect to API - Unknown error occurred: {e}."

url_check_msg

### auth/token

In [ ]:
token_url = f"{api_url}/db2/siteToken"
token_response = requests.request(method='GET', url=token_url)
token_response_data = token_response.json()
if not(token_response_data["error"]):
    anonymous_token = token_response_data['data']['token']
else:
    anonymous_token = ""
    print(f"error in response when acquiring token: {token_response.status}")

anonymous_token

In [ ]:
def get_anonymous_token(base_url):
    token_url = f"{base_url}/db2/siteToken"
    token_response = requests.request(method='GET', url=token_url)
    token_response_data = token_response.json()
    if not(token_response_data["error"]):
        anonymous_token = token_response_data['data']['token']
    else:
        anonymous_token = ""
        Warning(f"error in response when acquiring token: {token_response.status}")

    return(anonymous_token)

api_token = get_anonymous_token(api_url)
print(api_token)
if not api_token:
    print("no token, stop here")

### get stations

In [ ]:
token_header = {"Authorization": f"Bearer {anonymous_token}"}
station_list_url = f"{api_url}/db2/places?show=ewxstation"
station_list_response = requests.request(method='GET', url=station_list_url, headers = token_header )
station_list = station_list_container[0]['options']
active_station_list = [s for s in station_list if s['endDate'] == todays_date]
stations_df = pd.DataFrame(active_station_list)
stations_df

In [8]:
from enviroweather.ewxapi import EWXAPI
dir(EWXAPI)

SyntaxError: expected ':' (ewxapi.py, line 29)

### create map-able data

convert for mapping using GeoPandas

see https://eoda-dev.github.io/py-openlayers/concepts/geopandas/

In [ ]:
#

### Map with OpenLayers

see https://pypi.org/project/openlayers/

examples https://molab.marimo.io/github/marimo-team/gallery-examples/blob/main/notebooks/geo/earthquake.py/wasm